## 1. Configuration de l'environnement

In [ ]:
# Imports nécessaires
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import time

print("📦 Imports réalisés avec succès !")
print(f"🐼 Pandas version: {pd.__version__}")

In [ ]:
# Configuration des chemins (Docker)
import os

base_path = "/workspace/Brief_Starter_Pack"
orders_path = f"{base_path}/data/march-input/"
customers_path = f"{base_path}/data/march-input/customers.csv"

print("📂 Configuration:")
print(f"Base path: {base_path}")
print(f"Orders: {orders_path}")
print(f"Customers: {customers_path}")

# Vérifier l'existence
print(f"\n✅ Fichiers présents:")
print(f"Customers: {os.path.exists(customers_path)}")
print(f"Orders dir: {os.path.exists(orders_path)}")


## 2. Création de données d'exemple

In [ ]:
# 📊 CHARGEMENT DES VRAIES DONNÉES FRESHKART
# Utilisation des vraies données e-commerce au lieu de données simulées

print("🔍 Chargement des données FreshKart...")

# Chemins vers vos vraies données
base_path = "c:/Users/red59/Documents/Brief_Starter_Pack/Starter stack pour Data Engineers - Partie 1"
orders_path = f"{base_path}/data/march-input/"
customers_path = f"{base_path}/data/march-input/customers.csv"

# Vérification des fichiers disponibles
import os
if os.path.exists(orders_path):
    json_files = [f for f in os.listdir(orders_path) if f.startswith('orders_') and f.endswith('.json')]
    print(f"📋 Fichiers JSON trouvés: {len(json_files)}")
    print(f"📁 Chemin orders: {orders_path}")
else:
    print(f"⚠️  Chemin introuvable: {orders_path}")

if os.path.exists(customers_path):
    print(f"👥 Fichier clients: {customers_path}")
else:
    print(f"⚠️  Fichier clients introuvable: {customers_path}")

print("\n🎯 Nous allons utiliser ces vraies données pour les comparaisons Pandas vs PySpark !")

## 3. Comparaison Pandas vs PySpark - Création des DataFrames

In [ ]:
# 🐼 PANDAS - Chargement des clients FreshKart
print("🐼 PANDAS - Chargement des données FreshKart")

start_time = time.time()

try:
    # Charger les clients avec Pandas
    df_customers_pandas = pd.read_csv(customers_path)
    
    # Charger 3 fichiers JSON pour la démo (éviter surcharge)
    pattern = f"{orders_path}/orders_2025-03-*.json"
    import glob
    files = glob.glob(pattern)[:3]  # Seulement 3 fichiers pour la démo
    
    pandas_dataframes = []
    for file_path in files:
        df_temp = pd.read_json(file_path)
        pandas_dataframes.append(df_temp)
    
    if pandas_dataframes:
        df_orders_pandas = pd.concat(pandas_dataframes, ignore_index=True)
        pandas_creation_time = time.time() - start_time
        
        print(f"   ⏱️  Temps chargement: {pandas_creation_time:.4f}s")
        print(f"   👥 Clients: {len(df_customers_pandas):,} lignes")
        print(f"   📦 Commandes: {len(df_orders_pandas):,} lignes (3 fichiers JSON)")
        print(f"   💾 Mémoire: {(df_customers_pandas.memory_usage(deep=True).sum() + df_orders_pandas.memory_usage(deep=True).sum()) / 1024**2:.2f} MB")
        
        print("\n👀 Aperçu clients:")
        display(df_customers_pandas.head(3))
        
        print("\n👀 Aperçu commandes:")
        display(df_orders_pandas.head(3))
    else:
        print("⚠️  Aucun fichier JSON chargé")
        
except Exception as e:
    print(f"⚠️  Erreur chargement: {e}")
    print("💡 Vérifiez que les fichiers existent dans le bon chemin")

In [ ]:
# ⚡ PYSPARK - Chargement optimisé des données FreshKart
print("⚡ PYSPARK - Chargement des données FreshKart")

start_time = time.time()

try:
    # Charger TOUS les fichiers JSON en une seule opération !
    orders_pattern = f"{orders_path}/orders_2025-03-*.json"
    df_orders_spark = spark.read.option("multiline", "true").json(orders_pattern)
    
    # Charger les clients
    df_customers_spark = spark.read.option("header", "true").csv(customers_path)
    
    # Compter les lignes (déclenche l'exécution)
    orders_count = df_orders_spark.count()
    customers_count = df_customers_spark.count()
    spark_creation_time = time.time() - start_time
    
    print(f"   ⏱️  Temps chargement: {spark_creation_time:.4f}s")
    print(f"   👥 Clients: {customers_count:,} lignes")
    print(f"   📦 Commandes: {orders_count:,} lignes (TOUS les fichiers JSON!)")
    print(f"   📋 Colonnes commandes: {len(df_orders_spark.columns)}")
    
    print("\n📊 Schéma des commandes:")
    df_orders_spark.printSchema()
    
    print("\n📊 Schéma des clients:")
    df_customers_spark.printSchema()
    
except Exception as e:
    print(f"⚠️  Erreur chargement: {e}")
    print("💡 Vérifiez que les fichiers existent dans le bon chemin")
    
    # Fallback: créer des données d'exemple si pas de fichiers
    print("\n🔄 Création de données d'exemple pour continuer...")
    np.random.seed(42)
    n_records = 1000
    sample_data = {
        'order_id': range(1, n_records + 1),
        'customer_id': np.random.randint(1, 100, n_records),
        'amount': np.round(np.random.uniform(10, 500, n_records), 2)
    }
    df_orders_spark = spark.createDataFrame(pd.DataFrame(sample_data))
    print(f"   📦 Données d'exemple créées: {df_orders_spark.count()} lignes")

In [ ]:
# Aperçu des données PySpark FreshKart
print("👀 Aperçu des commandes PySpark:")
if 'df_orders_spark' in locals():

    df_orders_spark.show(5, truncate=False)    print("⚠️  DataFrame non disponible")

    else:

    print("\n👀 Aperçu des clients PySpark:")        df_orders_spark.groupBy("payment_status").count().show()

    if 'df_customers_spark' in locals():        print("\n💳 Statuts de paiement:")

        df_customers_spark.show(5)    if 'payment_status' in df_orders_spark.columns:

        

    # Statistiques rapides        df_orders_spark.groupBy("channel").count().show()

    print("\n📊 Statistiques des commandes:")    if 'channel' in df_orders_spark.columns:

## 4.5 Exemple pratique : Chargement de multiples fichiers JSON

Démonstration avec les vraies données FreshKart - 31 fichiers de commandes JSON

In [ ]:
# 📁 AVANTAGE PYSPARK : Chargement de multiples fichiers JSON en une seule opération !

# Chemin vers vos données FreshKart
base_path = "c:/Users/red59/Documents/Brief_Starter_Pack/Starter stack pour Data Engineers - Partie 1"
orders_path = f"{base_path}/data/march-input/"

print("🔍 Exploration des fichiers de commandes:")

# Lister les fichiers disponibles
import os
json_files = [f for f in os.listdir(orders_path) if f.startswith('orders_') and f.endswith('.json')]
json_files.sort()

print(f"📋 Fichiers JSON trouvés: {len(json_files)}")
print(f"📅 Premier fichier: {json_files[0] if json_files else 'Aucun'}")
print(f"📅 Dernier fichier: {json_files[-1] if json_files else 'Aucun'}")

# COMPARAISON : Pandas vs PySpark pour multiples JSON

In [ ]:
# 🐼 MÉTHODE PANDAS : Boucle pour charger chaque fichier (moins efficace)
import pandas as pd
import glob

print("🐼 PANDAS - Chargement avec boucle:")
start_time = time.time()

# Simuler l'approche Pandas classique
pandas_dataframes = []
pattern = f"{orders_path}/orders_2025-03-*.json"
files = glob.glob(pattern)

print(f"📁 Fichiers à charger: {len(files)}")

# Charger seulement les 3 premiers pour la démo (éviter le temps d'attente)
for i, file_path in enumerate(files[:3]):
    df_temp = pd.read_json(file_path)
    pandas_dataframes.append(df_temp)
    print(f"   📄 Fichier {i+1}: {len(df_temp)} lignes")

# Concaténation
if pandas_dataframes:
    df_pandas_combined = pd.concat(pandas_dataframes, ignore_index=True)
    pandas_time = time.time() - start_time
    
    print(f"⏱️  PANDAS Total: {pandas_time:.4f}s")
    print(f"📊 Résultat: {len(df_pandas_combined)} commandes (3 fichiers)")
else:
    print("⚠️  Aucune donnée chargée")

In [ ]:
# ⚡ MÉTHODE PYSPARK : Chargement intelligent en une seule opération !

print("⚡ PYSPARK - Chargement optimisé:")
start_time = time.time()

# Pattern matching pour tous les fichiers JSON
orders_pattern = f"{orders_path}/orders_2025-03-*.json"

try:
    # PySpark charge TOUS les fichiers JSON automatiquement !
    df_orders_spark = spark.read.option("multiline", "true").json(orders_pattern)
    
    # Compter les lignes (action qui déclenche l'exécution)
    total_orders = df_orders_spark.count()
    spark_time = time.time() - start_time
    
    print(f"⏱️  PYSPARK Total: {spark_time:.4f}s")
    print(f"📊 Résultat: {total_orders:,} commandes (TOUS les fichiers)")
    print(f"🚀 Speedup: PySpark charge TOUS les fichiers automatiquement !")
    
    # Aperçu du schéma
    print("\n📋 Schéma des commandes:")
    df_orders_spark.printSchema()
    
    # Quelques exemples
    print("\n👀 Aperçu des commandes:")
    df_orders_spark.show(5, truncate=False)
    
    # Statistiques rapides
    print("\n📊 Statistiques par canal:")
    df_orders_spark.groupBy("channel").count().show()
    
except Exception as e:
    print(f"⚠️  Erreur: {e}")
    print("💡 Vérifiez que les fichiers JSON existent dans le chemin spécifié")

In [ ]:
# 🔍 ANALYSE DES DONNÉES JSON : Explosion des items

print("🔍 Analyse avancée des commandes JSON avec PySpark:")

if 'df_orders_spark' in locals():
    # Vérifier la structure des items
    print("\n📦 Structure des items dans les commandes:")
    df_orders_spark.select("order_id", "items").show(3, truncate=False)
    
    # Exploser les items (comme dans le pipeline FreshKart)
    from pyspark.sql.functions import explode, col
    
    df_orders_exploded = df_orders_spark.withColumn("item", explode(col("items")))
    
    # Extraire les propriétés des items
    df_orders_items = df_orders_exploded.select(
        "order_id",
        "customer_id", 
        "channel",
        "payment_status",
        "created_at",
        col("item.sku").alias("item_sku"),
        col("item.qty").alias("item_qty"), 
        col("item.unit_price").alias("item_unit_price")
    )
    
    print(f"\n📊 Après explosion des items: {df_orders_items.count():,} lignes")
    print("\n👀 Aperçu des items explosés:")
    df_orders_items.show(10)
    
    # Analyse des statuts de paiement
    print("\n💳 Répartition des statuts de paiement:")
    df_orders_spark.groupBy("payment_status").count().orderBy("count", ascending=False).show()
    
    # Analyse des canaux
    print("\n📱 Répartition par canal:")
    df_orders_spark.groupBy("channel").count().show()
    
else:
    print("⚠️  DataFrame des commandes non disponible")

### 🎯 Avantages PySpark pour multiples JSON :

**✅ Avantages PySpark :**
- **Une seule commande** : `spark.read.json("pattern/*.json")` charge tout
- **Parallélisation automatique** : traitement simultané de tous les fichiers  
- **Schema inference** : détection automatique de la structure JSON
- **Lazy evaluation** : optimisation des opérations avant exécution
- **Gestion mémoire** : pas besoin de tout charger en RAM

**🐌 Limitations Pandas :**
- **Boucle manuelle** : `for file in files: pd.read_json(file)`
- **Concaténation nécessaire** : `pd.concat(dataframes)`
- **Mémoire limitée** : tout doit tenir en RAM
- **Pas d'optimisation** : chaque fichier traité séquentiellement

**📊 Résultat :** PySpark traite vos 31 fichiers JSON comme un seul dataset unifié !

## 4. Opérations de base - Sélection et Filtrage

In [ ]:
# 🐼 PANDAS - Sélection et filtrage sur les données FreshKart
print("🐼 PANDAS - Sélection et filtrage")

if 'df_orders_pandas' in locals() and 'df_customers_pandas' in locals():
    # Analyse des clients
    print("\n👥 Analyse des clients:")
    clients_by_segment = df_customers_pandas['segment'].value_counts()
    print(f"📋 Segments clients: {clients_by_segment.to_dict()}")
    
    # Filtrage des clients Premium
    premium_customers = df_customers_pandas[df_customers_pandas['segment'] == 'Premium']
    print(f"🎆 Clients Premium: {len(premium_customers)}")
    
    # Analyse des commandes (si colonnes disponibles)
    print("\n📦 Analyse des commandes:")
    print(f"📏 Total commandes: {len(df_orders_pandas)}")
    

    # Vérifier les colonnes disponibles    print("⚠️  Données Pandas non disponibles - utilisez les cellules précédentes")

    print(f"📊 Colonnes disponibles: {list(df_orders_pandas.columns)}")else:

            

    # Filtrage conditionnel selon les colonnes        print(f"💳 Commandes payées: {len(paid_orders)}")

    if 'channel' in df_orders_pandas.columns:        paid_orders = df_orders_pandas[df_orders_pandas['payment_status'] == 'paid']

        online_orders = df_orders_pandas[df_orders_pandas['channel'] == 'online']    if 'payment_status' in df_orders_pandas.columns:

        print(f"📱 Commandes online: {len(online_orders)}")    

In [ ]:
# ⚡ PYSPARK - Sélection et filtrage sur les données FreshKart
print("⚡ PYSPARK - Sélection et filtrage")

if 'df_orders_spark' in locals() and 'df_customers_spark' in locals():
    # Analyse des clients
    print("\n👥 Analyse des clients:")
    if 'segment' in df_customers_spark.columns:
        print("📋 Segments clients:")
        df_customers_spark.groupBy("segment").count().show()
        
        # Filtrage des clients Premium
        premium_customers = df_customers_spark.filter(col('segment') == 'Premium')
        print(f"🎆 Clients Premium: {premium_customers.count()}")
    
    # Analyse des commandes
    print("\n📦 Analyse des commandes:")
    total_orders = df_orders_spark.count()
    print(f"📏 Total commandes: {total_orders:,}")
    
    # Filtrage conditionnel selon les colonnes disponibles

    columns_available = df_orders_spark.columns    print("⚠️  Données PySpark non disponibles - exécutez les cellules de chargement")

    print(f"📊 Colonnes disponibles: {columns_available}")else:

            

    if 'channel' in columns_available:        ).show(5)

        print("\n📱 Répartition par canal:")            (col('payment_status') == 'paid')

        df_orders_spark.groupBy("channel").count().orderBy("count", ascending=False).show()            (col('channel') == 'online') & 

                df_orders_spark.filter(

        # Filtrage canal online        print("\n👀 Aperçu commandes online payées:")

        online_orders = df_orders_spark.filter(col('channel') == 'online')    if 'channel' in columns_available and 'payment_status' in columns_available:

        print(f"📱 Commandes online: {online_orders.count():,}")    # Aperçu des résultats

        

    if 'payment_status' in columns_available:        print(f"💳 Commandes payées: {paid_orders.count():,}")

        print("\n💳 Répartition par statut paiement:")        paid_orders = df_orders_spark.filter(col('payment_status') == 'paid')

        df_orders_spark.groupBy("payment_status").count().orderBy("count", ascending=False).show()        # Filtrage commandes payées
        

## 5. Agrégations et GroupBy

In [ ]:
# 🐼 PANDAS - Agrégations sur les données FreshKart
print("🐼 PANDAS - Agrégations")

if 'df_customers_pandas' in locals():
    start_time = time.time()
    
    # Agrégations sur les clients
    print("\n👥 Statistiques clients par segment:")
    customers_by_segment = df_customers_pandas.groupby('segment').agg({
        'customer_id': 'count',
        'email': 'count'
    }).rename(columns={'customer_id': 'count', 'email': 'emails'})
    
    pandas_agg_time = time.time() - start_time
    print(f"⏱️  Temps agrégation: {pandas_agg_time:.4f}s")

        print("⚠️  Données non disponibles - exécutez d'abord le chargement Pandas")

    display(customers_by_segment)else:

                print(f"Analyse des montants via colonne: {amount_col}")

    # Analyse des données géographiques si disponibles            amount_col = amount_cols[0]

    if 'country' in df_customers_pandas.columns:        if amount_cols:

        print("\n🌍 Répartition géographique:")        amount_cols = [col for col in df_orders_pandas.columns if 'amount' in col.lower() or 'price' in col.lower() or 'total' in col.lower()]

        geo_stats = df_customers_pandas['country'].value_counts().head(5)        # Chercher une colonne de montant

        display(geo_stats)        

            print(f"Colonnes commandes: {list(df_orders_pandas.columns)}")

    # Si on a des commandes, analyser les montants        print("\n📦 Analyse des commandes (si montants disponibles):")
    if 'df_orders_pandas' in locals() and len(df_orders_pandas) > 0:

In [ ]:
# ⚡ PYSPARK - Agrégations sur les données FreshKart
print("⚡ PYSPARK - Agrégations")

if 'df_customers_spark' in locals():
    start_time = time.time()
    
    # Agrégations sur les clients
    print("\n👥 Statistiques clients par segment:")
    customers_agg = df_customers_spark.groupBy('segment').agg(
        count('customer_id').alias('total_customers')
    ).orderBy('total_customers', ascending=False)
    
    # Déclencher l'exécution
    customers_results = customers_agg.collect()
    spark_agg_time = time.time() - start_time
    
    print(f"⏱️  Temps agrégation: {spark_agg_time:.4f}s")
    customers_agg.show()
    
    # Analyse géographique si disponible

    if 'country' in df_customers_spark.columns:    print("⚠️  Données non disponibles - exécutez d'abord le chargement PySpark")

        print("\n🌍 Top pays:")else:

        df_customers_spark.groupBy('country').count().orderBy('count', ascending=False).limit(5).show()            

                ).orderBy('total_orders', ascending=False).show()

    # Agrégations sur les commandes si disponibles                count('order_id').alias('total_orders')

    if 'df_orders_spark' in locals():            df_orders_spark.groupBy('payment_status').agg(

        print("\n📦 Statistiques des commandes:")            print("\n💳 Commandes par statut de paiement:")

        orders_count = df_orders_spark.count()        if 'payment_status' in df_orders_spark.columns:

        print(f"Total commandes: {orders_count:,}")        # Agrégation par statut de paiement si disponible

                

        # Agrégation par canal si disponible            ).orderBy('total_orders', ascending=False).show()

        if 'channel' in df_orders_spark.columns:                count('order_id').alias('total_orders')

            print("\n📱 Commandes par canal:")            df_orders_spark.groupBy('channel').agg(

## 6. Jointures et opérations avancées

In [ ]:
# Créer une table de référence des catégories
category_info = {
    'category': categories,
    'commission_rate': [0.05, 0.08, 0.03, 0.06, 0.07],
    'min_order': [50, 20, 10, 30, 25]
}

# DataFrame Pandas
df_categories_pandas = pd.DataFrame(category_info)
print("🐼 Table catégories Pandas:")
print(df_categories_pandas)

# DataFrame Spark
df_categories_spark = spark.createDataFrame(df_categories_pandas)
print("\n⚡ Table catégories Spark:")
df_categories_spark.show()

In [ ]:
# PANDAS - Jointure
start_time = time.time()
pandas_joined = df_pandas.merge(df_categories_pandas, on='category', how='inner')
pandas_join_time = time.time() - start_time

# Calcul commission
pandas_joined['commission'] = pandas_joined['amount'] * pandas_joined['commission_rate']

print(f"🐼 PANDAS Jointure: {pandas_join_time:.4f}s")
print(f"📊 Résultat: {pandas_joined.shape[0]} lignes")

# Aperçu
pandas_joined[['order_id', 'category', 'amount', 'commission_rate', 'commission']].head()

In [ ]:
# PYSPARK - Jointure
start_time = time.time()
spark_joined = df_spark.join(df_categories_spark, on='category', how='inner')

# Calcul commission
spark_joined = spark_joined.withColumn(
    'commission', 
    col('amount') * col('commission_rate')
)

# Exécution avec collect pour mesurer le temps
result_count = spark_joined.count()
spark_join_time = time.time() - start_time

print(f"⚡ PYSPARK Jointure: {spark_join_time:.4f}s")
print(f"📊 Résultat: {result_count} lignes")

# Aperçu
spark_joined.select('order_id', 'category', 'amount', 'commission_rate', 'commission').show(5)

## 7. Analyse comparative des performances

In [ ]:
# Résumé des performances
performance_data = {
    'Opération': ['Création DF', 'Agrégation', 'Jointure'],
    'Pandas (s)': [pandas_creation_time, pandas_agg_time, pandas_join_time],
    'PySpark (s)': [spark_creation_time, spark_agg_time, spark_join_time]
}

perf_df = pd.DataFrame(performance_data)
perf_df['Ratio PySpark/Pandas'] = perf_df['PySpark (s)'] / perf_df['Pandas (s)']

print("⚡ Comparaison des performances")
print("=" * 50)
print(perf_df.round(4))

print("\n📊 Observations:")
print(f"📦 Dataset size: {n_records:,} lignes")
print(f"🖥️  Mode: Local (single machine)")
print(f"⚙️  Cores utilisés: {sc.defaultParallelism}")

## 8. Conversion entre Pandas et PySpark

In [ ]:
# PySpark vers Pandas
print("🔄 Conversion PySpark → Pandas")
start_time = time.time()
spark_to_pandas = spark_joined.toPandas()
to_pandas_time = time.time() - start_time
print(f"⏱️  Temps: {to_pandas_time:.4f}s")
print(f"📊 Shape: {spark_to_pandas.shape}")

# Pandas vers PySpark
print("\n🔄 Conversion Pandas → PySpark")
start_time = time.time()
pandas_to_spark = spark.createDataFrame(spark_to_pandas)
to_spark_time = time.time() - start_time
print(f"⏱️  Temps: {to_spark_time:.4f}s")
print(f"📊 Lignes: {pandas_to_spark.count()}")

print("\n✅ Conversions réussies !")

## 9. Concepts clés à retenir

### 🔄 Lazy Evaluation
- PySpark utilise l'évaluation paresseuse
- Les transformations sont planifiées, pas exécutées immédiatement
- L'exécution se fait lors des actions (.show(), .collect(), .count())

### 📊 API Differences
- Pandas: `df.groupby()` → PySpark: `df.groupBy()`
- Pandas: `df[condition]` → PySpark: `df.filter(condition)`
- Pandas: `df['col']` → PySpark: `df.select('col')` ou `col('col')`

### ⚡ Performance
- PySpark excelle sur les gros volumes
- Pandas plus rapide sur petits datasets (< 1GB)
- PySpark parallelise automatiquement les opérations

### 🎯 Quand utiliser quoi ?
- **Pandas**: Prototypage, petites données, analyse interactive
- **PySpark**: Production, gros volumes, données distribuées

## 10. Exercices pratiques

### Exercice 1: Analysez les ventes par région
Calculez le chiffre d'affaires total et moyen par région avec PySpark

In [ ]:
# TODO: Votre code ici
# Indice: utilisez groupBy('region') et agg()

### Exercice 2: Trouvez les top 3 des catégories par commission
En utilisant le DataFrame jointé avec les commissions

In [ ]:
# TODO: Votre code ici
# Indice: groupBy, sum, orderBy, limit

### Exercice 3: Créez une nouvelle colonne avec la marge bénéficiaire
Supposons une marge de 20% sur le montant

In [ ]:
# TODO: Votre code ici
# Indice: withColumn et col()

## 🎉 Félicitations !

Vous avez terminé votre première session PySpark ! 

### 📚 Prochaines étapes :
1. 📖 Notebook 2: Transformations avancées et UDF
2. 🔧 Notebook 3: Optimisation et performance tuning
3. 🎯 Projets pratiques complets

### 🔗 Ressources utiles :
- [Documentation PySpark](https://spark.apache.org/docs/latest/api/python/)
- [Guide de migration Pandas→PySpark](../documentation/)
- [Exercices supplémentaires](../exercices_pratiques/)

In [ ]:
# Nettoyage - Fermer la SparkSession
spark.stop()
print("🛑 SparkSession fermée. À bientôt ! 🚀")

## 🔍 Problème de synchronisation Jupyter Lab

**❓ Pourquoi vous ne voyez que 2 notebooks au lieu de 3 ?**

Il y a 3 notebooks dans le projet VSCode :
- `01_introduction_pyspark.ipynb` ✅ (visible)
- `02_pyspark_avance.ipynb` ✅ (visible) 
- `03_migration_freshkart_pyspark.ipynb` ❌ (manquant dans Jupyter)

**🛠️ Solutions pour voir tous les notebooks :**

1. **Redémarrer Jupyter Lab :**
   ```bash
   # Dans WSL
   jupyter lab stop
   jupyter lab --ip=0.0.0.0 --port=8888 --no-browser
   ```

2. **Rafraîchir le navigateur :** Ctrl+F5 dans l'onglet Jupyter Lab

3. **Vérifier le répertoire :** Jupyter Lab doit être lancé depuis `/mnt/c/Users/red59/Documents/Brief_Starter_Pack/notebooks`

4. **Alternative :** Ouvrir directement le fichier manquant via le File Browser de Jupyter Lab

**📂 Structure complète du projet :**
```
notebooks/
├── 01_introduction_pyspark.ipynb    # Introduction et comparaisons
├── 02_pyspark_avance.ipynb          # Transformations avancées  
└── 03_migration_freshkart_pyspark.ipynb  # Migration complète FreshKart
```

## ✅ Résumé des modifications apportées

**🔄 Ce notebook a été adapté pour utiliser vos vraies données FreshKart :**

### 📊 Section 2 - Données réelles :
- ❌ ~~Génération de 10,000 commandes fictives~~
- ✅ **Chargement des vrais clients** (`customers.csv`)  
- ✅ **Chargement des vraies commandes** (31 fichiers JSON)

### 🔍 Section 4 - Opérations adaptées :
- ❌ ~~Filtrage sur `category` et `amount`~~
- ✅ **Analyse des segments clients** (Premium, Standard, etc.)
- ✅ **Filtrage par canal** (online, mobile, store)
- ✅ **Analyse des statuts de paiement** (paid, pending, failed)

### 📈 Section 5 - Agrégations réalistes :
- ❌ ~~Agrégations sur categories fictives~~
- ✅ **Statistiques par segment client**
- ✅ **Répartition géographique par pays**  
- ✅ **Analyse des commandes par canal et statut**

### 🎯 Avantages démontrés avec vos données :
1. **PySpark** : Charge les 31 JSON automatiquement avec `*.json` pattern
2. **Pandas** : Doit boucler sur chaque fichier et concaténer
3. **Performance** : PySpark traite tout en parallèle vs Pandas séquentiel

**▶️ Prochaines étapes :**
1. Exécuter ce notebook avec vos vraies données FreshKart
2. Observer la différence de performance Pandas vs PySpark  
3. Passer au notebook `02_pyspark_avance.ipynb` pour les transformations complexes
4. Finaliser avec `03_migration_freshkart_pyspark.ipynb` pour le pipeline complet